# Example Inference Notebook (Quick Check)

This notebook:
- loads one fold checkpoint
- runs quick validation check on train rows
- runs quick test inference and writes a preview submission


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "twitter_sentiment"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repo ready at:", REPO_DIR)


In [ ]:
import json
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer

import sys
sys.path.insert(0, str(Path("src").resolve()))

from dz2_causal.dataset import register_special_tokens
from dz2_causal.modeling import CausalExtractionModel
from dz2_causal.eval_utils import evaluate_dataframe_jaccard, predict_selected_text

CFG = json.loads(Path("config/kaggle_example_inference.json").read_text())
CFG


In [ ]:
train_df = pd.read_csv(CFG["train_csv"]).dropna(subset=["text", "selected_text"]).reset_index(drop=True)
test_df = pd.read_csv(CFG["test_csv"]).fillna("").reset_index(drop=True)
sub = pd.read_csv(CFG["sample_submission_csv"]).fillna("")

fold_id = int(CFG["fold_for_inference"])
ckpt_path = Path(CFG["output_dir"]) / f"model_fold{fold_id}.pt"
print("Checkpoint:", ckpt_path)
assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"],
    use_fast=True,
    trust_remote_code=CFG.get("trust_remote_code", False),
)
model = CausalExtractionModel(
    model_name=CFG["model_name"],
    trust_remote_code=CFG.get("trust_remote_code", False),
)
register_special_tokens(tokenizer, model=model.lm)
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state["model_state_dict"], strict=True)
model.to(device)
model.eval()
print("Model loaded.")


In [ ]:
quick_val = train_df.sample(n=min(CFG["quick_eval_rows"], len(train_df)), random_state=CFG["seed"]).reset_index(drop=True)
quick_eval = evaluate_dataframe_jaccard(
    df=quick_val,
    model=model,
    tokenizer=tokenizer,
    prompt_text=CFG["prompt_text"],
    device=device,
    max_new_tokens=CFG["max_new_tokens"],
)
print("Quick validation Jaccard:", quick_eval["mean_jaccard"])


In [ ]:
n_test = min(CFG["test_rows_for_fast_check"], len(test_df))
preview = test_df.iloc[:n_test].copy().reset_index(drop=True)

preds = []
for row in preview.itertuples(index=False):
    pred = predict_selected_text(
        model=model,
        tokenizer=tokenizer,
        prompt=CFG["prompt_text"],
        tweet=str(row.text),
        sentiment=str(row.sentiment),
        device=device,
        max_new_tokens=CFG["max_new_tokens"],
    )
    preds.append(pred)

preview_sub = sub.iloc[:n_test].copy()
preview_sub["selected_text"] = preds
preview_path = Path(CFG["output_dir"]) / "submission_preview.csv"
preview_sub.to_csv(preview_path, index=False)
print("Saved preview:", preview_path)
preview_sub.head()


In [ ]:
# Optional full submission (set True when needed)
RUN_FULL_SUBMISSION = False

if RUN_FULL_SUBMISSION:
    full_preds = []
    for row in test_df.itertuples(index=False):
        pred = predict_selected_text(
            model=model,
            tokenizer=tokenizer,
            prompt=CFG["prompt_text"],
            tweet=str(row.text),
            sentiment=str(row.sentiment),
            device=device,
            max_new_tokens=CFG["max_new_tokens"],
        )
        full_preds.append(pred)

    sub_full = sub.copy()
    sub_full["selected_text"] = full_preds
    full_path = Path(CFG["output_dir"]) / "submission_full.csv"
    sub_full.to_csv(full_path, index=False)
    print("Saved full submission:", full_path)
